# 04. Label-shuffle check

The class labels are randomly permuted once, which removes any real link between genes and classes. Class counts are unchanged, so the always-NEG rate stays 44.7%.

- With selection **inside** each round, accuracy should fall to about the always-NEG rate or below. Anything clearly higher would point to a leak in the pipeline.
- With selection **outside** the loop (fitted on all patients), the shuffled labels can still look learnable, because the selected genes were chosen to fit those labels. This shows the size of the leakage effect when there is no real signal.

50 rounds are used for each run to keep the run time reasonable.

In [1]:
import os
import sys
sys.path.insert(0, '../src')
import numpy as np
import pandas as pd
from pipeline import CLASSES, evaluate, load_data

X, y = load_data()
y_shuffled = pd.Series(np.random.default_rng(0).permutation(y.to_numpy()), index=y.index)
print('class counts unchanged:', y_shuffled.value_counts().to_dict())
print('patients keeping their true label by chance:', int((y_shuffled == y).sum()), 'of', len(y))

# the two runs take about 20 minutes; saved scores are reused if present (delete them to recompute)
if os.path.exists('../results/shuffle_outside_scores.csv'):
    inside = pd.read_csv('../results/shuffle_inside_scores.csv')
    outside = pd.read_csv('../results/shuffle_outside_scores.csv')
else:
    inside, _ = evaluate(X, y_shuffled, n_rounds=50, seed=0, leak=False)
    outside, _ = evaluate(X, y_shuffled, n_rounds=50, seed=0, leak=True)
    inside.to_csv('../results/shuffle_inside_scores.csv', index=False)
    outside.to_csv('../results/shuffle_outside_scores.csv', index=False)

class counts unchanged: {'NEG': 42, 'BCR/ABL': 37, 'ALL1/AF4': 10, 'E2A/PBX1': 5}
patients keeping their true label by chance: 36 of 94


In [2]:
for df in (inside, outside):
    df['workflow'] = df['selection'] + '_' + df['model']

cols = ['accuracy', 'balanced_accuracy']
print('always-NEG reference accuracy:', round(y.value_counts(normalize=True).max(), 3), '| chance balanced accuracy: 0.25')
print()
print('mean over all 24 workflows')
print(pd.DataFrame({'inside': inside[cols].mean(), 'outside': outside[cols].mean()}).round(3))
print()
print('best single workflow (mean over rounds)')
print(pd.DataFrame({'inside': inside.groupby('workflow')[cols].mean().max(),
                    'outside': outside.groupby('workflow')[cols].mean().max()}).round(3))

always-NEG reference accuracy: 0.447 | chance balanced accuracy: 0.25

mean over all 24 workflows
                   inside  outside
accuracy            0.380    0.502
balanced_accuracy   0.233    0.379

best single workflow (mean over rounds)
                   inside  outside
accuracy            0.448    0.578
balanced_accuracy   0.250    0.501


In [3]:
res = inside.groupby('workflow')[cols].mean().join(outside.groupby('workflow')[cols].mean(),
                                                    lsuffix='_inside', rsuffix='_outside')
res.sort_values('accuracy_outside', ascending=False).round(3)

,accuracy_inside,balanced_accuracy_inside,accuracy_outside,balanced_accuracy_outside
workflow,,,,
s1_rf_mtry5,0.386,0.240,0.578,0.442
s1_rf_mtry15,0.380,0.236,0.568,0.437
s2_rf_mtry5,0.393,0.245,0.564,0.434
s2_rf_mtry15,0.384,0.240,0.558,0.430
s2_svm_C100_g0.001,0.357,0.224,0.553,0.437
s1_svm_C100_g0.001,0.358,0.225,0.553,0.437
s1_svm_C1_g0.01,0.403,0.237,0.530,0.309
s2_svm_C1_g0.01,0.401,0.236,0.530,0.309
s1_svm_C100_g0.01,0.350,0.225,0.516,0.501


## Notes

**Result**

- With selection inside each round, shuffled labels give a mean accuracy of 0.38 over the 24 workflows, and the best single workflow reaches 0.448, which is the always-NEG rate (0.447). Mean balanced accuracy is 0.23, close to the chance level of 0.25 for four classes. The pipeline therefore finds no signal where none exists, which is the expected behaviour when selection is truly confined to the training patients. This supports the baseline accuracies in notebook 02.
- With selection fitted on all patients (outside the loop), the same shuffled labels give a mean accuracy of 0.50 and a mean balanced accuracy of 0.38; the best workflow reaches 0.578 accuracy and 0.50 balanced accuracy. There is no real signal, so this entire gain is leakage: the selected genes were picked to fit the shuffled labels, including for the held-out patients.

**Limits**

- One shuffle (seed 0) and 50 rounds per run, so the numbers are indicative. A different permutation would move them by a few points.
- Mean accuracy inside the loop (0.38) is below the always-NEG rate because the models predict the shuffled minority classes some of the time; this is expected for a model with no signal and is not a sign of a fault.